# 02 -- Anchoring evidence

The design brief asserts three things as priors from the literature. This
notebook tests all three on the panel actually in hand, because the Stage 2
corrections only earn their place if the pathologies they target are present
here.

**A. Targets are a stable multiple of prevailing price.** Analysts set a target
at roughly a fixed multiple of spot and revise after the stock moves. If true,
the level of a target is mostly a restatement of the current price.

**B. Analysts revise after the stock moves, not before.** A firm whose revisions
load heavily on the trailing 20-day return is following price, and carries
little independent information.

**C. The level is close to uninformative; the derivative is weakly informative.**

Test C is a **single-ticker time-series** test. The literature's result is
*cross-sectional*: rank a universe each month and compare deciles. One ticker
cannot reproduce that and cannot refute it. What it can do is show whether this
particular panel is so uninformative that there is no point proceeding.

All of it lives in `src/diagnostics/anchoring.py` and is unit-tested.

In [ ]:
import sys, warnings
from datetime import date, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src import config
from src.store import pit, writer

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

TICKER = "AAPL"
ASOF = date.today()

# 'strict' is the only mode whose output is safe to publish. On a store built
# from a single snapshot it returns nothing for past as-of dates, which is the
# truthful answer -- see src/store/pit.py. Switch to ASSUME_VENDOR_HISTORY only
# with that assumption in mind.
PIT_MODE = pit.ASSUME_VENDOR_HISTORY

con = writer.connect(config.DB_PATH, read_only=True)
prov = pit.provenance(con, TICKER)
prov


In [ ]:
from src.diagnostics.anchoring import run_anchoring_study, render_anchoring_report

study = run_anchoring_study(con, TICKER, ASOF, pit_mode=PIT_MODE)
print(render_anchoring_report(study))


## A. The PT/spot ratio, by firm

This is the single most important picture in the project. If each firm's ratio
clusters tightly around its own value, and those values differ across firms,
then the level of an implied return is mostly telling you *which firm wrote it*.

That is exactly what the Stage 2a correction removes -- and exactly why
averaging raw targets across firms does not cancel anything.

In [ ]:
study.firm_ratios


In [ ]:
try:
    import matplotlib.pyplot as plt
    fr = study.firm_ratios.sort_values("median")
    fig, ax = plt.subplots(figsize=(11, max(3.5, 0.28 * len(fr))))
    ax.errorbar(fr["median"], range(len(fr)),
                xerr=fr["sd"].fillna(0), fmt="o", ms=4, capsize=2, lw=1)
    ax.axvline(1.0, ls="--", c="grey", lw=1)
    ax.set_yticks(range(len(fr)))
    ax.set_yticklabels(fr["analyst_firm"], fontsize=8)
    ax.set_xlabel("price target / spot at action")
    ax.set_title(f"{TICKER}: each firm's habitual multiple of spot (median +/- 1sd)")
    plt.tight_layout()
except ImportError:
    print("matplotlib not installed; the table above carries the same information")


In [ ]:
# The same thing as one distribution: how far from 1.0 does the street sit?
from src.model.debias import attach_implied_returns

panel = pit.price_targets_asof(con, TICKER, ASOF, pit_mode=PIT_MODE)
history = pit.price_history_asof(con, TICKER, ASOF)
df = attach_implied_returns(panel, history)
ratio = (df.loc[df["usable"], "price_target_used"] / df.loc[df["usable"], "spot_pit"])

print(f"n = {len(ratio)}")
print(f"share of targets ABOVE the prevailing price: {(ratio > 1).mean():.1%}")
print(ratio.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))


## B. Do revisions follow the stock?

Regress the log change in a firm's target on the stock's trailing 20-day return.
A positive loading means the firm is restating the recent past. A loading near
1.0 means the firm is doing essentially nothing but re-anchoring.

This is the Stage 3 "timeliness" measure, computed here because it is also the
cleanest evidence for the brief's second design constraint.

In [ ]:
study.follow_by_firm


## C. Level versus derivative against the realised outcome

Month-end snapshots of the panel, each paired with what the stock actually did
over the following 12 months. Consecutive rows overlap by eleven months, so the
regressions use Newey-West standard errors -- a naive t-statistic here is
inflated several-fold.

**A |t| below about 2 is the expected result.** That is the brief's prior, and
it is the reason the composite must be validated against spot in Stage 8 before
it is allowed to emit anything.

In [ ]:
study.snapshots.tail(24)


In [ ]:
for t in (study.level_test, study.revision_test):
    print(f"{t['label']}")
    if t["n"] and np.isfinite(t.get("beta", np.nan)):
        print(f"   beta {t['beta']:+.3f}   se {t['se']:.3f}   t {t['t']:+.2f}"
              f"   R2 {t['r2']:.3f}   n {t['n']}   [{t['note']}]")
    else:
        print(f"   {t['note']}")
    print()


In [ ]:
try:
    import matplotlib.pyplot as plt
    s = study.snapshots.dropna(subset=["level_raw", "fwd_return"])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(s["level_raw"], s["fwd_return"], s=14, alpha=0.7)
    axes[0].set_xlabel("median implied return (level)")
    axes[0].set_ylabel("realised 12m return")
    axes[0].set_title("level vs outcome")
    r = study.snapshots.dropna(subset=["revision_breadth", "fwd_return"])
    axes[1].scatter(r["revision_breadth"], r["fwd_return"], s=14, alpha=0.7, color="darkorange")
    axes[1].set_xlabel("revision breadth (derivative)")
    axes[1].set_title("derivative vs outcome")
    for ax in axes:
        ax.axhline(0, ls="--", c="grey", lw=0.8)
    plt.tight_layout()
except ImportError:
    print("matplotlib not installed; the table above carries the same information")


## What this does and does not establish

Establishes, if A and B come back positive: the anchoring the Stage 2
corrections target is present in this panel, so the corrections are removing
something real rather than shuffling noise.

Does **not** establish: that the de-biased panel predicts anything. Nothing in
this notebook compares a forecast against spot. That is Stage 8, it has not been
built, and until it runs the correct prior is that the distribution carries no
information.